# ML-06 — Signal Audit: Do the Flags Hold?

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
!git clone https://github.com/Abhaykr45/flyrank-ml-internship.git
%cd flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 176, done.
remote: Counting objects: 100% (176/176), done.
remote: Compressing objects: 100% (146/146), done.
remote: Total 176 (delta 73), reused 77 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (176/176), 2.12 MiB | 2.66 MiB/s, done.
Resolving deltas: 100% (73/73), done.
/content/flyrank-ml-internship


In [7]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())

Dataset loaded successfully!
Dataset shape: (30000, 44)

First 5 rows:


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [8]:
features_to_check = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position"
]

print("Required signal columns:")

for col in features_to_check:
    if col in df.columns:
        print("✓", col)
    else:
        print("✗ MISSING:", col)

Required signal columns:
✓ search_volume
✓ competition
✓ cpc
✓ word_count
✓ char_count
✓ content_age_days
✓ days_since_last_update
✓ ctr
✓ avg_position


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [9]:
# ML-06 — Section 1: Distribution Audit

features_to_check = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position"
]

distribution_stats = df[features_to_check].describe().T

print("Distribution Summary:")
display(distribution_stats)

Distribution Summary:


,count,mean,std,min,25%,50%,75%,max
search_volume,27532.0,158.882391,1518.270825,0.0,0.0,10.00,20.00,74000.00
competition,27532.0,0.146954,0.285241,0.0,0.0,0.00,0.13,1.00
cpc,27532.0,0.485342,2.101560,0.0,0.0,0.00,0.00,100.36
word_count,22301.0,3107.760325,1452.382598,8.0,2413.0,2877.00,3666.00,9546.00
char_count,22301.0,20665.277835,10115.344042,40.0,15644.0,19116.00,24011.00,111158.00
content_age_days,30000.0,256.167800,132.707930,90.0,132.0,236.00,333.00,564.00
days_since_last_update,30000.0,46.098300,42.078709,1.0,20.0,20.00,104.00,373.00
ctr,30000.0,0.510733,3.279162,0.0,0.0,0.07,0.29,100.00
avg_position,30000.0,16.342380,15.216790,0.0,6.2,10.80,22.30,245.00


In [10]:
# Compare mean, median and maximum values to identify heavy tails

distribution_check = pd.DataFrame({
    "mean": df[features_to_check].mean(),
    "median": df[features_to_check].median(),
    "max": df[features_to_check].max(),
    "missing": df[features_to_check].isna().sum(),
    "skew": df[features_to_check].skew()
})

display(distribution_check)

,mean,median,max,missing,skew
search_volume,158.882391,10.00,74000.00,2468,26.016250
competition,0.146954,0.00,1.00,2468,2.043852
cpc,0.485342,0.00,100.36,2468,13.737601
word_count,3107.760325,2877.00,9546.00,7699,0.937935
char_count,20665.277835,19116.00,111158.00,7699,1.551197
content_age_days,256.167800,236.00,564.00,0,0.489000
days_since_last_update,46.098300,20.00,373.00,0,1.161283
ctr,0.510733,0.07,100.00,0,17.444252
avg_position,16.342380,10.80,245.00,0,1.984214


### Distribution Audit — Interpretation

I inspected the distributions of the nine candidate signals before testing
their relationship with the target.

The signals operate on different numerical scales, and some variables show
skewed or heavy-tailed behaviour. For this reason, I considered the median
alongside the mean rather than interpreting averages alone.

Missing-value counts were also inspected before the signal tests. These
distribution checks are descriptive and are used to understand the data
before making directional claims about individual signals.

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [11]:
# Check target distribution

print("Target classes:")
print(df["trend_direction"].value_counts(dropna=False))

Target classes:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


In [12]:
# ML-06 — Three Signal Tests

signals = [
    "content_age_days",
    "days_since_last_update",
    "avg_position"
]

for signal in signals:
    print("\n" + "=" * 60)
    print("SIGNAL:", signal)
    print("=" * 60)

    result = (
        df.groupby("trend_direction")[signal]
          .agg(["count", "mean", "median"])
          .round(3)
    )

    display(result)


SIGNAL: content_age_days


,count,mean,median
trend_direction,,,
down,16262,236.179,216.0
flat,1152,245.890,231.0
new,2236,238.718,279.0
stable,5962,295.440,300.0
up,4388,288.479,291.5



SIGNAL: days_since_last_update


,count,mean,median
trend_direction,,,
down,16262,49.246,20.0
flat,1152,37.454,20.0
new,2236,22.337,20.0
stable,5962,50.062,22.0
up,4388,43.426,22.0



SIGNAL: avg_position


,count,mean,median
trend_direction,,,
down,16262,15.936,11.3
flat,1152,11.102,6.6
new,2236,9.912,0.0
stable,5962,16.333,11.4
up,4388,22.513,15.3


### Signal Test Verdicts

#### Signal #1 — Content Age

**Question:** Does content age clearly separate declining pages from pages moving upward?

**Observed result:**  
The mean `content_age_days` is approximately 236 days for `down` pages and
288 days for `up` pages. Stable pages are also relatively old at approximately
295 days on average.

**Verdict: MIXED**

Content age differs across trend groups, but older content is not consistently
associated with decline. Age alone is therefore not a sufficient refresh signal.

---

#### Signal #2 — Days Since Last Update

**Question:** Are declining pages generally further from their most recent update?

**Observed result:**  
`down` pages average approximately 49 days since the last update, compared with
approximately 43 days for `up` pages and 50 days for `stable` pages.

**Verdict: MIXED**

There is some directional evidence that declining pages may have gone longer
without an update than upward-moving pages, but the stable group has a similar
or higher value. Update recency should therefore be combined with other signals.

---

#### Signal #3 — Average Position

**Question:** Does average search position differ across trend-direction groups?

**Observed result:**  
Mean `avg_position` differs substantially across the groups: approximately
15.9 for `down`, 11.1 for `flat`, 9.9 for `new`, 16.3 for `stable`, and
22.5 for `up`.

**Verdict: CONFIRMED (as a differentiating signal), with caution**

Average position clearly varies across the observed trend groups. However,
the relationship is not simply monotonic, and this analysis is observational.
It should be treated as a useful model/review signal rather than evidence that
average position causes a particular trend direction.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [13]:
# ML-06 — Section 3: Find available flag-related columns

flag_columns = [
    col for col in df.columns
    if "flag" in col.lower()
    or "refresh" in col.lower()
    or "opportunity" in col.lower()
]

print("Possible flag-related columns:")
for col in flag_columns:
    print("-", col)

Possible flag-related columns:


In [14]:
# ML-06 — Section 3: Refresh-related proxy test

# Compare trend direction for more recently updated vs older-update pages
audit_df = df[["days_since_last_update", "trend_direction"]].dropna().copy()

median_update_gap = audit_df["days_since_last_update"].median()

audit_df["update_group"] = audit_df["days_since_last_update"].apply(
    lambda x: "Longer since update" if x > median_update_gap
    else "More recently updated"
)

flag_test = pd.crosstab(
    audit_df["update_group"],
    audit_df["trend_direction"],
    normalize="index"
).round(3)

print("Median days since last update:", median_update_gap)
print("\nTrend-direction proportions by update-recency group:")
display(flag_test)

Median days since last update: 20.0

Trend-direction proportions by update-recency group:


trend_direction,down,flat,new,stable,up
update_group,,,,,
Longer since update,0.546,0.024,0.012,0.250,0.168
More recently updated,0.539,0.051,0.130,0.153,0.127


### Flag-Linked Test Interpretation

The dataset does not contain an explicit column named as a FlyRank flag, so I
did not invent one. Instead, I tested `days_since_last_update` as a
refresh-related proxy signal.

I compared pages with longer update gaps against more recently updated pages
and examined their observed `trend_direction` distributions.

This test is directional rather than causal. Update recency may contribute
useful information to a content-review rule, but it should not be treated as
sufficient evidence that updating a page will cause its search performance to
improve. The signal should be combined with other content and search signals
and reviewed by a human.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

### Practical Takeaway

The observed signal tests suggest that content age, update recency, and average
position can provide useful directional information, but no single signal is
strong enough to determine a content action by itself.

A content team should combine these signals to prioritize pages for human
review rather than automatically deciding that a page must be refreshed.

These findings are observational and decision-support oriented; they do not
show that changing any individual signal will cause search performance to
improve.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.